# LedgerLock: AP Invoice Fraud Detection POC

This notebook demonstrates the end-to-end pipeline for detecting fraudulent or anomalous AP invoices in logistics & cold-chain shipping using the LedgerLock POC.

## 1. Project Structure and File Generation

The project is organized as follows:

```
fraud-ap-poc/
├── data/                # Sample data files (invoices, vendors, BOLs)
├── notebooks/           # Jupyter notebooks for experimentation
├── src/                 # Core Python modules (parser, rules, report, main)
├── tests/               # Unit tests
├── requirements.txt     # Python dependencies
├── README.md            # Project overview and quickstart
├── .gitignore           # Ignore Python/Jupyter/OS artifacts
```

Starter files and directories are generated as described in the project brief.

## 2. Populate Sample Data CSVs

Sample data is provided in the `data/` directory:
- `invoices_sample.csv`: Contains realistic AP invoice rows, including duplicates, unknown vendor, bank mismatch, and unusual amounts.
- `vendor_master.csv`: Vendor master data for validation.
- `bol_sample.csv`: Bill of Lading records for optional BOL checks.

Let's preview the sample data.

In [ ]:
import pandas as pd

invoices = pd.read_csv('../data/invoices_sample.csv')
vendors = pd.read_csv('../data/vendor_master.csv')
bols = pd.read_csv('../data/bol_sample.csv')

print('Sample Invoices:')
display(invoices)
print('Vendor Master:')
display(vendors)
print('BOL Sample:')
display(bols)

## 3. Implement Data Parsers

The `src/parser.py` module provides functions to load invoices, vendors, and BOLs from CSV files into pandas DataFrames. It also includes a stub for future PDF parsing.

Example usage:

In [ ]:
from src import parser

invoices = parser.load_invoices('../data/invoices_sample.csv')
vendors = parser.load_vendors('../data/vendor_master.csv')
bols = parser.load_bols('../data/bol_sample.csv')

invoices.head()

## 4. Implement Core Fraud Rules

The `src/rules.py` module implements the following fraud checks as pure functions:
- **flag_duplicates**: Detects duplicate invoice ID + vendor.
- **flag_bank_mismatch**: Flags invoices where the bank account does not match the vendor master.
- **flag_unknown_vendor**: Flags vendors not in the master list, with fuzzy matching suggestions.
- **flag_unusual_amounts**: Flags invoices with amounts > 2× vendor's historical average.
- **flag_bol_mismatch** (optional): Flags invoices with BOLs not in the BOL master list.

Each function returns a DataFrame of flagged invoices with reasons.

In [ ]:
from src import rules

flags = []
flags.append(rules.flag_duplicates(invoices))
flags.append(rules.flag_bank_mismatch(invoices, vendors))
flags.append(rules.flag_unknown_vendor(invoices, vendors))
flags.append(rules.flag_unusual_amounts(invoices, vendors))
flags.append(rules.flag_bol_mismatch(invoices, bols))

all_flags = pd.concat(flags, ignore_index=True)

print(f"Flagged {len(all_flags)} invoices. Reasons:")
display(all_flags)
all_flags['reason'].value_counts()

## 5. Report Generation Utilities

The `src/report.py` module provides functions to export flagged invoices to Excel or CSV, including a summary sheet for Excel output.

In [ ]:
from src import report

# Save to Excel
report.export_report(all_flags, '../data/fraud_report.xlsx')
# Save to CSV
report.export_report(all_flags, '../data/fraud_report.csv')

print('Fraud report saved to data/fraud_report.xlsx and data/fraud_report.csv')

## 6. CLI Entry Point

The `src/main.py` module provides a command-line interface to run the full pipeline:

```bash
python -m src.main --invoices data/invoices_sample.csv --vendors data/vendor_master.csv --bol data/bol_sample.csv --out fraud_report.xlsx
```

This loads the data, runs all fraud rules, outputs a report, and prints a summary to stdout.

## 7. Unit Tests for Fraud Rules

Unit tests are provided in `tests/test_rules.py` using pytest. These cover:
- Duplicate detection
- Unknown vendor detection
- Unusual amount detection

Example test (for duplicates):

In [ ]:
import pandas as pd
from src import rules

def test_flag_duplicates():
    df = pd.DataFrame({
        'invoice_id': ['A', 'A', 'B'],
        'vendor_name': ['X', 'X', 'Y'],
        'bank_account': ['1', '1', '2'],
        'invoice_date': ['2025-01-01']*3,
        'amount': [100, 100, 200],
        'bol_id': ['B1', 'B1', 'B2']
    })
    flagged = rules.flag_duplicates(df)
    assert len(flagged) == 2
    assert all(flagged['reason'] == 'Duplicate invoice ID and vendor')

## 8. Jupyter Notebook: End-to-End Pipeline Demo

This notebook demonstrates the full pipeline:
1. Load sample data
2. Run all fraud rules
3. Display flagged results
4. Save a sample fraud report
5. Visualize flag counts by type

In [ ]:
import plotly.express as px

flag_counts = all_flags['reason'].value_counts().reset_index()
flag_counts.columns = ['reason', 'count']
fig = px.bar(flag_counts, x='reason', y='count', title='Fraud Flags by Type')
fig.show()

## 9. Suggest Additional Logistics-Specific Checks

Potential future fraud/anomaly checks for logistics AP:
- **Repeated detention charges**: Flag vendors/invoices with excessive detention fees.
- **Weekend/holiday invoice anomalies**: Flag invoices dated on weekends/holidays.
- **Accessorial frequency outliers**: Detect unusual frequency of accessorial charges (e.g., lumper, layover).
- **Rapid vendor bank account changes**: Track and flag vendors with frequent bank changes.
- **Sanctions/OFAC checks**: Cross-reference vendors against OFAC SDN lists.
- **Telematics/BOL mismatch**: Cross-check BOLs with GPS/telematics data for route/timing anomalies.

`TODO:` These can be added as new functions in `src/rules.py` for future releases.